# SafeLens Montgomery — Weather Context (Open-Meteo)

**Zero setup — no API key, no account, completely free forever.**

Run the SQL in Supabase SQL Editor first:
```sql
CREATE TABLE IF NOT EXISTS public.weather_context (
  id           UUID NOT NULL DEFAULT gen_random_uuid(),
  date         DATE UNIQUE NOT NULL,
  avg_temp_f   FLOAT,
  max_temp_f   FLOAT,
  min_temp_f   FLOAT,
  wind_mph     FLOAT,
  precip_mm    FLOAT,
  description  TEXT,
  is_heat_day  BOOLEAN DEFAULT FALSE,
  is_rain_day  BOOLEAN DEFAULT FALSE,
  is_storm_day BOOLEAN DEFAULT FALSE,
  created_at   TIMESTAMPTZ DEFAULT NOW(),
  CONSTRAINT weather_context_pkey PRIMARY KEY (id)
);
CREATE INDEX IF NOT EXISTS idx_weather_date ON public.weather_context(date DESC);
```

In [9]:
# ── CELL 1: Install + Connect ──────────────────────────────────────────────
!pip install supabase requests pandas --quiet

import requests
import pandas as pd
from datetime import datetime, timezone, timedelta

# HOW TO ADD YOUR KEYS:
#   1. Click the 🔑 key icon in the LEFT sidebar of Colab
#   2. Add secret: Name=SUPABASE_URL  Value=https://vkveikbxhlrneejexuyl.supabase.co
#   3. Add secret: Name=SUPABASE_KEY  Value=sb_publishable_...
#   4. Toggle 'Notebook access' ON for both

try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_KEY')
    print('✅ Credentials loaded from Colab Secrets')
except Exception:
    import os
    SUPABASE_URL = os.getenv('SUPABASE_URL')
    SUPABASE_KEY = os.getenv('SUPABASE_KEY')
    print('✅ Credentials loaded from environment variables')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(
        '❌ Missing credentials!\n'
        '   Add SUPABASE_URL and SUPABASE_KEY to Colab Secrets\n'
        '   Click the 🔑 icon in the left sidebar'
    )

from supabase import create_client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

DRY_RUN = False  # ← flip to False when ready to write

try:
    supabase.table('incidents').select('id').limit(1).execute()
    print('✅ Supabase connected!')
except Exception as e:
    print(f'❌ Connection error: {e}')

print(f'DRY_RUN = {DRY_RUN}')

✅ Credentials loaded from Colab Secrets
✅ Supabase connected!
DRY_RUN = False


In [10]:
# ── CELL 2: Fetch from Open-Meteo ─────────────────────────────────────────
#
# LEARNING — Open-Meteo vs NOAA:
#   NOAA = 2-step grid discovery + station lookup + hourly aggregation
#   Open-Meteo = one URL, pass lat/lng, get daily data back directly
#   Same data quality, 10x simpler. No key needed ever.
#
# WMO Weather codes: 0=Clear, 1-3=Cloudy, 45/48=Fog,
#   51-67=Rain, 71-77=Snow, 80-82=Showers, 95-99=Thunderstorm

MONTGOMERY_LAT = 32.3617
MONTGOMERY_LNG = -86.2792
DAYS_BACK      = 90

end_date   = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=DAYS_BACK)

print(f'📡 Fetching {DAYS_BACK} days of weather from Open-Meteo...')
print(f'   {start_date} → {end_date}\n')

r = requests.get(
    'https://archive-api.open-meteo.com/v1/archive',
    params={
        'latitude':          MONTGOMERY_LAT,
        'longitude':         MONTGOMERY_LNG,
        'start_date':        str(start_date),
        'end_date':          str(end_date),
        'daily': [
            'temperature_2m_max',
            'temperature_2m_min',
            'temperature_2m_mean',
            'precipitation_sum',
            'windspeed_10m_max',
            'weathercode',
        ],
        'timezone':          'America/Chicago',
        'temperature_unit':  'celsius',
        'windspeed_unit':    'kmh',
        'precipitation_unit':'mm',
    },
    timeout=30
)
r.raise_for_status()
raw = r.json()

days = raw.get('daily', {}).get('time', [])
print(f'✅ {len(days)} days returned')
print(f'Fields: {list(raw.get("daily", {}).keys())}')

📡 Fetching 90 days of weather from Open-Meteo...
   2025-12-09 → 2026-03-09

✅ 91 days returned
Fields: ['time', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'precipitation_sum', 'windspeed_10m_max', 'weathercode']


In [11]:
# ── CELL 3: Normalize → weather_context schema ─────────────────────────────

WMO = {
    0:'Clear sky', 1:'Mainly clear', 2:'Partly cloudy', 3:'Overcast',
    45:'Foggy', 48:'Icy fog',
    51:'Light drizzle', 53:'Drizzle', 55:'Heavy drizzle',
    61:'Light rain', 63:'Rain', 65:'Heavy rain',
    71:'Light snow', 73:'Snow', 75:'Heavy snow', 77:'Snow grains',
    80:'Rain showers', 81:'Rain showers', 82:'Heavy rain showers',
    85:'Snow showers', 86:'Heavy snow showers',
    95:'Thunderstorm', 96:'Thunderstorm w/ hail', 99:'Heavy thunderstorm',
}

def c_to_f(c):   return round((c*9/5)+32, 1)  if c is not None else None
def kmh_to_mph(k): return round(k*0.621371, 1) if k is not None else None

daily    = raw['daily']
records  = []

for i, date in enumerate(daily['time']):
    max_f  = c_to_f(daily['temperature_2m_max'][i])
    min_f  = c_to_f(daily['temperature_2m_min'][i])
    avg_f  = c_to_f(daily['temperature_2m_mean'][i])
    wind   = kmh_to_mph(daily['windspeed_10m_max'][i])
    precip = daily['precipitation_sum'][i] or 0.0
    code   = int(daily['weathercode'][i]) if daily['weathercode'][i] is not None else 0
    desc   = WMO.get(code, 'Unknown')

    records.append({
        'date':         date,
        'avg_temp_f':   avg_f,
        'max_temp_f':   max_f,
        'min_temp_f':   min_f,
        'wind_mph':     wind,
        'precip_mm':    round(float(precip), 2),
        'description':  desc,
        'is_heat_day':  bool(max_f  and max_f  >= 95),
        'is_rain_day':  bool(precip and precip > 5),
        'is_storm_day': bool((precip and precip > 20) or (wind and wind > 40)),
    })

df = pd.DataFrame(records)
print(f'📊 {len(records)} daily records built')
print(f'Date range:  {df["date"].min()} → {df["date"].max()}')
print(f'Avg temp:    {df["avg_temp_f"].mean():.1f}°F')
print(f'Max temp:    {df["max_temp_f"].max():.1f}°F')
print(f'Heat days:   {df["is_heat_day"].sum()}')
print(f'Rain days:   {df["is_rain_day"].sum()}')
print(f'Storm days:  {df["is_storm_day"].sum()}')
print()
display(df.head(10)[['date','max_temp_f','precip_mm','description','is_heat_day','is_rain_day','is_storm_day']])

📊 91 daily records built
Date range:  2025-12-09 → 2026-03-09
Avg temp:    52.3°F
Max temp:    82.4°F
Heat days:   0
Rain days:   16
Storm days:  2



,date,max_temp_f,precip_mm,description,is_heat_day,is_rain_day,is_storm_day
0,2025-12-09,51.6,0.0,Mainly clear,False,False,False
1,2025-12-10,62.4,0.0,Overcast,False,False,False
2,2025-12-11,54.5,0.0,Overcast,False,False,False
3,2025-12-12,65.8,0.0,Mainly clear,False,False,False
4,2025-12-13,66.2,0.1,Light drizzle,False,False,False
5,2025-12-14,61.5,1.0,Drizzle,False,False,False
6,2025-12-15,43.7,0.0,Clear sky,False,False,False
7,2025-12-16,52.2,0.0,Overcast,False,False,False
8,2025-12-17,63.5,0.0,Overcast,False,False,False
9,2025-12-18,65.3,7.5,Light rain,False,True,False


In [12]:
# ── CELL 4: Write to Supabase ──────────────────────────────────────────────
# REMINDER: Run the CREATE TABLE SQL in Supabase SQL Editor first!

def write_batch(recs, table, on_conflict=None, chunk=500):
    if DRY_RUN:
        print(f'  🧪 DRY RUN — {len(recs)} records ready for "{table}"')
        return
    for i in range(0, len(recs), chunk):
        c = recs[i:i+chunk]
        try:
            q = supabase.table(table)
            (q.upsert(c, on_conflict=on_conflict) if on_conflict else q.upsert(c)).execute()
            print(f'  ✅ {min(i+chunk, len(recs))}/{len(recs)} → "{table}"')
        except Exception as e:
            print(f'  ❌ {e}')

print(f'Writing {len(records)} weather records...')
write_batch(records, 'weather_context', on_conflict='date')

if not DRY_RUN:
    result = supabase.table('weather_context').select('id', count='exact').execute()
    print(f'\n✅ weather_context: {result.count} rows total')

Writing 91 weather records...
  ✅ 91/91 → "weather_context"

✅ weather_context: 91 rows total


In [13]:
# ── CELL 5: Claude Prompt Pattern ─────────────────────────────────────────
# Copy this into your /api/narrative route (Next.js backend)
# Matches your narratives table schema exactly

def build_narrative_prompt(neighborhood_name, incidents, news_articles, weather_today):
    """
    Builds Claude prompt for narratives.content field.
    weather_today = one row from weather_context table.
    """
    weather_block = ''
    if weather_today:
        flags = []
        if weather_today.get('is_heat_day'):  flags.append('HEAT DAY ≥95°F')
        if weather_today.get('is_rain_day'):  flags.append('RAIN DAY')
        if weather_today.get('is_storm_day'): flags.append('STORM DAY')
        weather_block = f"""
WEATHER (use to contextualize spikes — never to alarm):
  {weather_today.get('description')} | {weather_today.get('max_temp_f')}°F high
  Precip: {weather_today.get('precip_mm')}mm | Wind: {weather_today.get('wind_mph')}mph
  Flags: {', '.join(flags) if flags else 'Normal conditions'}
"""
    return f"""You are SafeLens Montgomery — a responsible community safety AI.
Generate a calm, factual 2-3 sentence safety narrative for {neighborhood_name}.

INCIDENTS ({len(incidents)}):
{chr(10).join(f'- {i.get("type")} ({i.get("occurred_at","")[:10]})' for i in incidents[:8])}

NEWS ({len(news_articles)} articles):
{chr(10).join(f'- {a.get("headline")}' for a in news_articles[:4])}
{weather_block}
Rules: Be calm. If weather flags exist, use them to explain spikes.
End with one actionable safety tip. No addresses or names."""

# Preview with today's weather
if records:
    sample = build_narrative_prompt(
        'Oak Park',
        [{'type':'NOISE_COMPLAINT','occurred_at':records[0]['date']},
         {'type':'ABANDONED_VEHICLE','occurred_at':records[0]['date']}],
        [{'headline':'MPD increases patrols in Oak Park area'}],
        records[0]
    )
    print('📋 SAMPLE CLAUDE PROMPT:')
    print('='*50)
    print(sample)

📋 SAMPLE CLAUDE PROMPT:
You are SafeLens Montgomery — a responsible community safety AI.
Generate a calm, factual 2-3 sentence safety narrative for Oak Park.

INCIDENTS (2):
- NOISE_COMPLAINT (2025-12-09)
- ABANDONED_VEHICLE (2025-12-09)

NEWS (1 articles):
- MPD increases patrols in Oak Park area

WEATHER (use to contextualize spikes — never to alarm):
  Mainly clear | 51.6°F high
  Precip: 0.0mm | Wind: 6.3mph
  Flags: Normal conditions

Rules: Be calm. If weather flags exist, use them to explain spikes.
End with one actionable safety tip. No addresses or names.


In [14]:
# ── CELL 6: Final Table Verification ──────────────────────────────────────

print('📊 ALL TABLE COUNTS')
print('='*45)

for table in ['incidents','news_articles','neighborhoods','narratives',
              'resident_reports','weather_context','source_reputation',
              'narrative_feedback']:
    try:
        res = supabase.table(table).select('id', count='exact').execute()
        print(f'  {table:<25} {(res.count or 0):>8} rows')
    except Exception as e:
        print(f'  {table:<25} ❌ {str(e)[:35]}')

print()
print('✅ Weather pipeline complete!')
print(f'   {len(records)} days loaded into weather_context')
print('   Claude can now contextualize incident spikes with real weather data')

📊 ALL TABLE COUNTS
  incidents                   279528 rows
  news_articles                 1003 rows
  neighborhoods                   22 rows
  narratives                      26 rows
  resident_reports                 2 rows
  weather_context                 91 rows
  source_reputation                2 rows
  narrative_feedback               4 rows

✅ Weather pipeline complete!
   91 days loaded into weather_context
   Claude can now contextualize incident spikes with real weather data
